# 01 — Exploratory Data Analysis (EDA)

**Objective:** Understand the synthetic IoT dataset structure, visualise signal patterns, compare normal vs anomalous sequences, and validate generation quality before modelling.

**Sections:**
1. Generate the dataset
2. Dataset overview (shape, class balance)
3. Per-sensor signal visualisation (normal vs anomaly types)
4. Statistical comparison: normal vs anomaly
5. Correlation matrix
6. Save to `data/raw/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from src.data.generator import IoTDataGenerator

sns.set_style("darkgrid")
SAVE_DIR = "../data/raw"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Imports OK")

## 1. Generate Dataset

In [ ]:
gen = IoTDataGenerator(
    n_sensors=50,
    seq_length=60,
    n_sequences=40_000,
    anomaly_ratio=0.05,
    seed=42,
)

print("Generating dataset (this may take ~30s)…")
X, y = gen.generate_dataset()

print(f"\nDataset generated:")
print(f"  X shape : {X.shape}")
print(f"  y shape : {y.shape}")
print(f"  dtype   : {X.dtype}")
print(f"  Normal  : {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")
print(f"  Anomaly : {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)")

## 2. Class Balance & Validation

In [ ]:
assert X.shape == (40_000, 60, 50), f"Unexpected shape: {X.shape}"
assert abs(y.mean() - 0.05) < 0.01, f"Unexpected anomaly ratio: {y.mean():.3f}"
print("✅ Shape validation passed")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Normal", "Anomaly"], [(y==0).sum(), (y==1).sum()],
       color=["#2196F3", "#F44336"], alpha=0.85)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
for i, v in enumerate([(y==0).sum(), (y==1).sum()]):
    ax.text(i, v + 50, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/eda_class_balance.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Signal Visualisation — Normal vs Anomaly Types

In [ ]:
ANOMALY_TYPES = ["spike", "drift", "shift", "periodic", "silence"]
SENSOR_TYPES  = {"Vibration": 0, "Temperature": 1, "Pressure": 2, "RPM": 3}

normal_seq = gen.generate_normal_sequence()

fig, axes = plt.subplots(len(ANOMALY_TYPES), len(SENSOR_TYPES),
                          figsize=(16, 14), sharex=True)

for row, atype in enumerate(ANOMALY_TYPES):
    anomaly_seq = gen.inject_anomaly(normal_seq.copy(), anomaly_type=atype)
    for col, (sname, sidx) in enumerate(SENSOR_TYPES.items()):
        ax = axes[row][col]
        ax.plot(normal_seq[:, sidx],  color="#2196F3", alpha=0.7, lw=1.2, label="Normal")
        ax.plot(anomaly_seq[:, sidx], color="#F44336", alpha=0.8, lw=1.4, label=f"{atype}")
        if row == 0:
            ax.set_title(sname, fontsize=12, fontweight="bold")
        if col == 0:
            ax.set_ylabel(atype.capitalize(), fontsize=10, rotation=90)
        if row == 0 and col == 0:
            ax.legend(fontsize=8)

fig.suptitle("Normal vs Anomaly Signals — All Types × Sensor Categories",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
os.makedirs("../logs", exist_ok=True)
plt.savefig("../logs/eda_signal_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Statistical Comparison: Normal vs Anomaly

In [ ]:
X_normal  = X[y == 0]
X_anomaly = X[y == 1]

# Flatten to (N*T, F) for feature-wise statistics
X_n_flat = X_normal.reshape(-1, 50)
X_a_flat = X_anomaly.reshape(-1, 50)

print("Feature-wise statistics (first 10 sensors):")
print(f"{'Sensor':<10} {'Normal μ':>10} {'Normal σ':>10} {'Anomaly μ':>10} {'Anomaly σ':>10}")
print("-" * 52)
for i in range(10):
    print(f"  {i:<8} {X_n_flat[:,i].mean():>10.3f} {X_n_flat[:,i].std():>10.3f} "
          f"{X_a_flat[:,i].mean():>10.3f} {X_a_flat[:,i].std():>10.3f}")

# Boxplot for sensor 0
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, sidx in enumerate([0, 1, 2, 3]):
    data = [X_n_flat[:, sidx], X_a_flat[:, sidx]]
    bp = axes[col].boxplot(data, labels=["Normal", "Anomaly"],
                            patch_artist=True)
    bp["boxes"][0].set_facecolor("#2196F3")
    bp["boxes"][1].set_facecolor("#F44336")
    axes[col].set_title(f"Sensor {sidx} ({list(SENSOR_TYPES.keys())[sidx]})")
    axes[col].set_ylabel("Value")

plt.suptitle("Normal vs Anomaly Value Distribution per Sensor Type",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/eda_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Sensor Correlation Matrix (Normal Data)

In [ ]:
import pandas as pd

# Use mean across timesteps per sequence to get per-sequence feature summary
X_n_mean = X_normal.mean(axis=1)  # (N, 50)
corr_matrix = np.corrcoef(X_n_mean[:2000].T)  # subsample for speed

# Show a 10×10 block (first 10 sensors)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_matrix[:10, :10], annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=[f"S{i}" for i in range(10)],
            yticklabels=[f"S{i}" for i in range(10)])
ax.set_title("Sensor Correlation Matrix (Normal, first 10 sensors)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/eda_correlation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Note: Low cross-sensor correlation exposes individual sensor anomalies more clearly.")

## 6. Save to `data/raw/`

In [ ]:
np.save(f"{SAVE_DIR}/X.npy", X)
np.save(f"{SAVE_DIR}/y.npy", y)
print(f"✅ Saved:")
print(f"   {SAVE_DIR}/X.npy  {X.shape}")
print(f"   {SAVE_DIR}/y.npy  {y.shape}")
print("\n→ NEXT STEP: Run notebook 02_preprocessing.ipynb")